# XGBoost Model SHAP Analysis

## Import Libraries and Setup

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, mean_squared_error
from sklearn.inspection import permutation_importance
import pickle
import os
import warnings
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
warnings.filterwarnings('ignore')

# Set random state
SEP = os.sep
random_state = 21

print("Libraries imported successfully!")

## Load and Prepare Data

In [ ]:
# Load data
print("Loading data...")
df = pd.read_excel("PGM_CBFV_01_norm_feature.xlsx").sample(frac=1, random_state=random_state, ignore_index=True)


print("\nFirst 5 rows of data:")
display(df.head())

# Define features and target
y_prop = "Mass_Change"
feat_names = ["avg_number_of_valence_electrons", "EN_Pauling", "r_asym", 
              "H_chem", "H_el", "Time", "Coh_E", ]

X = np.array(df[feat_names])
Y = np.array(df[y_prop])

print(f"\nX shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"\nFeatures: {feat_names}")
print(f"Target: {y_prop}")


## Create Output Directories

In [ ]:
# Create directories for saving results
directories = [
    "trained_mod_CBFV_xgboost",
    "xgboost_parity_plots",
    "xgboost_results"
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"✓ Created directory: {directory}")

print("\nAll directories ready!")

## K-Fold Cross-Validation Training

In [ ]:
print("=" * 70)
print("TESTING PARAMETER WITH K-FOLD CV ")
print("=" * 70)

# Best parameter
param_set = {
    'n_estimators': [2000],  
    'max_depth': [8],  
    'learning_rate': [0.01],
    'min_child_weight': [1],
    'subsample': [0.75],
    'colsample_bytree': [0.75],  
    'gamma': [0],
    'reg_alpha': [0], 
    'reg_lambda': [1.5],
}
all_param_set_results = []

# K-Fold configuration
Kfold_val = 5

# Function to train a single fold 
def train_fold(fold_data):
    """Train a single fold - CPU version"""
    x_train, x_test, y_train, y_test, param_set, fold_idx, set_idx = fold_data
    
    # Create model
    xgb_params = {
        **param_set,
        'random_state': random_state,
        'objective': 'reg:squarederror',
        'tree_method': 'hist',   
        'n_jobs': -1            
    }
    
    # Train using sklearn API 
    xgb_mod = xgb.XGBRegressor(**xgb_params)
    xgb_mod.fit(
        x_train, y_train,
        eval_set=[(x_train, y_train), (x_test, y_test)],
        verbose=False
    )
    
    # Predictions
    y_train_pred = xgb_mod.predict(x_train)
    y_test_pred = xgb_mod.predict(x_test)
    
    return (y_train_pred, y_test_pred, xgb_mod)

for set_idx, param_set in enumerate(top_param_sets, 1):
    print(f"\n{'='*70}")
    print(f"BEST PARAMETER SET")
    print(f"{'='*70}")
    print("Parameters:")
    for param, value in param_set.items():
        print(f"  {param}: {value}")
    
    # Initialize K-Fold
    kf = KFold(n_splits=Kfold_val, shuffle=True, random_state=random_state)
    
    # Initialize performance metrics storage
    metrics = {
        'train_mae': [], 'test_mae': [],
        'train_mse': [], 'test_mse': [],
        'train_rmse': [], 'test_rmse': [],
        'train_mape': [], 'test_mape': [],
        'train_r2': [], 'test_r2': []
    }
    
    k_count = 1
    all_predictions = {'train': [], 'test': [], 'y_train': [], 'y_test': []}
    
   
    set_start_time = time.time()
    
    for train_indices, test_indices in kf.split(X):
        print(f"\n  Fold {k_count}/{Kfold_val}...", end="")
        
        # Split data
        x_train, x_test = X[train_indices], X[test_indices]
        y_train, y_test = Y[train_indices], Y[test_indices]
        
        # Train fold 
        fold_start = time.time()
        y_train_pred, y_test_pred, xgb_mod = train_fold(
            (x_train, x_test, y_train, y_test, param_set, k_count, set_idx)
        )
        fold_time = time.time() - fold_start
        
        # Store predictions
        all_predictions['train'].extend(y_train_pred)
        all_predictions['test'].extend(y_test_pred)
        all_predictions['y_train'].extend(y_train)
        all_predictions['y_test'].extend(y_test)
        
        # Calculate metrics
        metrics['train_mae'].append(round(mean_absolute_error(y_train, y_train_pred), 6))
        metrics['test_mae'].append(round(mean_absolute_error(y_test, y_test_pred), 6))
        
        train_mse = mean_squared_error(y_train, y_train_pred)
        test_mse = mean_squared_error(y_test, y_test_pred)
        metrics['train_mse'].append(round(train_mse, 6))
        metrics['test_mse'].append(round(test_mse, 6))
        
        metrics['train_rmse'].append(round(np.sqrt(train_mse), 6))
        metrics['test_rmse'].append(round(np.sqrt(test_mse), 6))
        
        metrics['train_mape'].append(round(mean_absolute_percentage_error(y_train, y_train_pred), 4))
        metrics['test_mape'].append(round(mean_absolute_percentage_error(y_test, y_test_pred), 4))
        
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        metrics['train_r2'].append(round(train_r2, 4))
        metrics['test_r2'].append(round(test_r2, 4))
        
        # Save model
        model_savename = f"trained_mod_CBFV_xgboost{SEP}xgb-mod-set{set_idx:02d}-K{k_count}.pkl"
        pickle.dump(xgb_mod, open(model_savename, 'wb'))
        
        print(f" MAE: {metrics['test_mae'][-1]:.6f}, R²: {test_r2:.4f} ({fold_time:.2f}s)")
        k_count += 1
    
    set_time = time.time() - set_start_time
    print(f"\n  Parameter set {set_idx} completed in: {set_time:.2f} seconds")
    
    # Calculate summary statistics
    print(f"\n  Performance Summary:")
    print("  " + "-"*66)
    
    evaluation_metrics = ['mae', 'mse', 'rmse', 'mape', 'r2']
    set_summary = {'param_set': set_idx}
    
    for metric in evaluation_metrics:
        train_key = f'train_{metric}'
        test_key = f'test_{metric}'
        
        train_mean = np.mean(metrics[train_key])
        train_std = np.std(metrics[train_key])
        test_mean = np.mean(metrics[test_key])
        test_std = np.std(metrics[test_key])
        
        set_summary[f'train_{metric}_mean'] = train_mean
        set_summary[f'train_{metric}_std'] = train_std
        set_summary[f'test_{metric}_mean'] = test_mean
        set_summary[f'test_{metric}_std'] = test_std
        
        print(f"  {metric.upper()}:")
        print(f"    Train: {train_mean:.6f} ± {train_std:.6f}")
        print(f"    Test:  {test_mean:.6f} ± {test_std:.6f}")
    
    all_param_set_results.append(set_summary)
    
    # Save detailed results
    results_df = pd.DataFrame(metrics)
    results_df.index = [f'Fold_{i+1}' for i in range(Kfold_val)]
    results_df.to_csv(f'xgboost_results/cv_results_set{set_idx:02d}.csv')
    
    # Save feature importance 
    try:
        importance_scores = xgb_mod.feature_importances_
        
        feature_importance_df = pd.DataFrame({
            'Feature': feat_names,
            'Importance': importance_scores
        }).sort_values('Importance', ascending=False)
        
        feature_importance_df.to_csv(
            f'xgboost_results/feature_importance_set{set_idx:02d}.csv',
            index=False
        )
    except Exception as e:
        print(f"\n  Warning: Could not extract feature importance: {e}")
    
    # Store for plotting in next cell
    locals()[f'metrics_set{set_idx}'] = metrics
    locals()[f'predictions_set{set_idx}'] = all_predictions
    locals()[f'params_set{set_idx}'] = param_set


print("\n" + "=" * 70)
print("K-FOLD CROSS-VALIDATION COMPLETED")
print("=" * 70)

## Generate Parity Plots 

In [ ]:
print("=" * 70)
print("GENERATING PARITY PLOT")
print("=" * 70)

# Calculate statistics
train_r2_mean = np.mean(cv_metrics['train_r2'])
test_r2_mean = np.mean(cv_metrics['test_r2'])
train_mae_mean = np.mean(cv_metrics['train_mae'])
test_mae_mean = np.mean(cv_metrics['test_mae'])

# Create parity plot
fig, ax = plt.subplots(figsize=(10, 10))

# Scatter plots
ax.scatter(
    cv_predictions['y_train'], cv_predictions['train'],
    alpha=0.6, s=60,
    label=f"Train (R\u00b2={train_r2_mean:.3f}, MAE={train_mae_mean:.4f})",
    edgecolors='black', linewidths=0.5
)

ax.scatter(
    cv_predictions['y_test'], cv_predictions['test'],
    alpha=0.6, s=60,
    label=f"Test (R\u00b2={test_r2_mean:.3f}, MAE={test_mae_mean:.4f})",
    edgecolors='black', linewidths=0.5
)

# Perfect prediction line
y_min, y_max = min(Y), max(Y)
ax.plot(
    [y_min, y_max], [y_min, y_max],
    'k--', lw=2.5,
    label='Perfect Prediction', alpha=0.7
)

# Formatting 
ax.set_xlabel("Experimental Mass Change", fontsize=14, fontweight='bold')
ax.set_ylabel("Predicted Mass Change", fontsize=14, fontweight='bold')

ax.set_title(
    f"Parity Plot - Best Parameter Set [CPU]\n"
    f"n_est={best_param_set['n_estimators']}, "
    f"max_d={best_param_set['max_depth']}, "
    f"lr={best_param_set['learning_rate']}",
    fontsize=14,
    fontweight='bold'
)

ax.legend(fontsize=11, loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.savefig(
    'xgboost_parity_plots/parity_plot_best.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

print("\u2713 Saved: xgboost_parity_plots/parity_plot_best.png")


## Generate Detailed Metrics Plots

In [ ]:
print("=" * 70)
print("GENERATING DETAILED METRICS PLOTS")
print("=" * 70)

for set_idx in range(1, top_n_sets + 1):
    metrics = locals()[f'metrics_set{set_idx}']
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    metric_titles = ['MAE', 'MSE', 'RMSE', 'MAPE', 'R² Score']
    evaluation_metrics = ['mae', 'mse', 'rmse', 'mape', 'r2']
    colors_train = plt.cm.Blues(0.7)
    colors_test = plt.cm.Oranges(0.7)
    
    for idx, metric in enumerate(evaluation_metrics):
        train_key = f'train_{metric}'
        test_key = f'test_{metric}'
        
        x_pos = np.arange(Kfold_val)
        width = 0.35
        
        bars1 = axes[idx].bar(x_pos - width/2, metrics[train_key], width, 
                              label='Train', alpha=0.8, color=colors_train, 
                              edgecolor='black', linewidth=1.2)
        bars2 = axes[idx].bar(x_pos + width/2, metrics[test_key], width, 
                              label='Test', alpha=0.8, color=colors_test,
                              edgecolor='black', linewidth=1.2)
        
        # Add value labels
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                             f'{height:.4f}',
                             ha='center', va='bottom', fontsize=8)
        
        axes[idx].set_xlabel('Fold', fontsize=11, fontweight='bold')
        axes[idx].set_ylabel(metric_titles[idx], fontsize=11, fontweight='bold')
        axes[idx].set_title(f'{metric_titles[idx]} Across Folds', 
                           fontsize=12, fontweight='bold')
        axes[idx].set_xticks(x_pos)
        axes[idx].set_xticklabels([f'K{i+1}' for i in range(Kfold_val)])
        axes[idx].legend(fontsize=10)
        axes[idx].grid(True, alpha=0.3, axis='y', linestyle='--')
    
    fig.delaxes(axes[5])
    
    plt.suptitle(f'Cross-Validation Metrics - Best Parameter Set', 
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'xgboost_results/metrics_comparison_set{set_idx:02d}.png', 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved: xgboost_results/metrics_comparison_set{set_idx:02d}.png")

print("\nAll metrics plots generated!")

In [ ]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

# Find best parameter set (based on lowest test MAE)
best_set_idx = comparison_df['test_mae_mean'].idxmin() + 1
best_set_params = top_param_sets[best_set_idx - 1]
best_row = comparison_df[comparison_df['param_set'] == best_set_idx].iloc[0]

print(f"\nBest Performing Parameter Set: Set {best_set_idx}")
print("-" * 70)

print("\nParameters:")
for param, value in best_set_params.items():
    print(f"  {param}: {value}")

print(f"\nPerformance Metrics:")
print(f"  Test MAE:  {best_row['test_mae_mean']:.6f} ± {best_row['test_mae_std']:.6f}")
print(f"  Test RMSE: {best_row['test_rmse_mean']:.6f} ± {best_row['test_rmse_std']:.6f}")
print(f"  Test MAPE: {best_row['test_mape_mean']:.6f} ± {best_row['test_mape_std']:.6f}")
print(f"  Test R²:   {best_row['test_r2_mean']:.6f} ± {best_row['test_r2_std']:.6f}")


# Save summary report
with open('xgboost_results/FINAL_SUMMARY.txt', 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("XGBoost Model Training - Final Summary (CPU-ONLY)\n")
    f.write("=" * 70 + "\n\n")
    
    f.write("Execution Configuration:\n")
    f.write("  Mode: CPU\n")
    f.write("  Tree Method: hist\n")
    f.write("  n_jobs: -1 (All CPU cores)\n")
    f.write(f"  Total Training Time: {search_time/60:.2f} minutes\n")
    
    f.write(f"\nBest Performing Parameter Set: Set {best_set_idx}\n")
    f.write("-" * 70 + "\n")
    f.write("Parameters:\n")
    for param, value in best_set_params.items():
        f.write(f"  {param}: {value}\n")
    
    f.write(f"\nPerformance Metrics:\n")
    f.write(f"  Test MAE:  {best_row['test_mae_mean']:.6f} ± {best_row['test_mae_std']:.6f}\n")
    f.write(f"  Test RMSE: {best_row['test_rmse_mean']:.6f} ± {best_row['test_rmse_std']:.6f}\n")
    f.write(f"  Test MAPE: {best_row['test_mape_mean']:.6f} ± {best_row['test_mape_std']:.6f}\n")
    f.write(f"  Test R²:   {best_row['test_r2_mean']:.6f} ± {best_row['test_r2_std']:.6f}\n")
    
    f.write("\n" + "=" * 70 + "\n")
    
    f.write("=" * 70 + "\n\n")
    f.write(comparison_df.to_string(index=False))

print("\n✓ Saved: xgboost_results/FINAL_SUMMARY.txt")

print("\n" + "=" * 70)
print("TRAINING COMPLETE! ")
print("=" * 70)
print(f"\nTotal training time: {search_time/60:.2f} minutes")
print("\nGenerated Files:")
print(f"  - {top_n_sets} parity plots in 'xgboost_parity_plots/' folder")
print(f"  - {top_n_sets} detailed metric files in 'xgboost_results/' folder")
print(f"  - Comparison results in 'xgboost_results/' folder")
print(f"  - {top_n_sets * Kfold_val} trained models in 'trained_mod_CBFV_xgboost/' folder")
print("  - Summary report: 'xgboost_results/FINAL_SUMMARY.txt'")
print("=" * 70)

# Optional: clear RAM (CPU-friendly cleanup)
import gc
gc.collect()
print("\n✓ CPU memory cleaned (garbage collection)")

## Per-Dopant-System SHAP Analysis

In [ ]:
print("=" * 70)
print("CELL 16 – PER-DOPANT-SYSTEM SHAP ANALYSIS")
print("=" * 70)

try:
    import shap
    print(f"  shap version: {shap.__version__}")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap
    print(f"  shap installed and imported: {shap.__version__}")

os.makedirs('xgboost_results/shap_per_dopant', exist_ok=True)

BASE_ELEMENTS   = ['Ni', 'Al']
DOPANT_ELEMENTS = ['Pt', 'Pd', 'Ir', 'Rh']


print("\nBuilding SHAP TreeExplainer on full-data model ...")
explainer = shap.TreeExplainer(full_lc_model)

print("Computing SHAP values for all training rows ...")
shap_values_all = explainer.shap_values(full_train_X)
print(f"  SHAP matrix shape: {shap_values_all.shape}")

unique_comps = sorted(set(full_train_alloys))
print(f"\nAll unique compositions in training pool ({len(unique_comps)}):")
for c in unique_comps:
    print(f"  {c:40s} — {(full_train_alloys == c).sum()} rows")

import re

def get_dopants_in_alloy(alloy_name, dopants=DOPANT_ELEMENTS):
    found = []
    for d in dopants:
        if re.search(r'(?<![A-Z])' + d + r'(?![a-z])', alloy_name):
            found.append(d)
    return found if found else ['NiAl_base']


dopant_to_comps = {d: [] for d in DOPANT_ELEMENTS + ['NiAl_base']}
for comp in unique_comps:
    for dop in get_dopants_in_alloy(comp):
        dopant_to_comps[dop].append(comp)
dopant_to_comps = {k: v for k, v in dopant_to_comps.items() if v}

print("\nDopant-system grouping:")
for dop, comps in dopant_to_comps.items():
    print(f"  {dop:12s}: {comps}")


shap_summary = {}
for comp in unique_comps:
    mask = full_train_alloys == comp
    shap_summary[comp] = np.abs(shap_values_all[mask]).mean(axis=0)


dopant_system_summary = {}

for dopant, comps_in_group in dopant_to_comps.items():
    n_alloys = len(comps_in_group)
    print(f"\n{'─'*60}")
    print(f"  Dopant group: {dopant}  ({n_alloys} compositions)")
    print(f"{'─'*60}")

    
    group_mask = np.zeros(len(full_train_alloys), dtype=bool)
    for comp in comps_in_group:
        group_mask |= (full_train_alloys == comp)

    sv_group  = shap_values_all[group_mask]   
    x_group   = full_train_X[group_mask]
    n_rows_grp = group_mask.sum()

    mean_abs_group = np.abs(sv_group).mean(axis=0)
    dopant_system_summary[dopant] = mean_abs_group

    print(f"    Total rows in group: {n_rows_grp}")


    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    
    plt.sca(axes[0])
    shap.summary_plot(sv_group, x_group,
                      feature_names=feat_names,
                      show=False, plot_size=None)
    axes[0].set_title(
        f'Dopant: {dopant}  |  SHAP Beeswarm (group avg, n={n_rows_grp} rows)',
        fontsize=11, fontweight='bold')

    
    ax_bar = axes[1]
    order  = np.argsort(mean_abs_group)[::-1]
    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(feat_names)))
    ax_bar.barh([feat_names[i] for i in order],
                mean_abs_group[order],
                color=colors, edgecolor='k', linewidth=0.4)
    ax_bar.set_xlabel('Mean |SHAP value|', fontsize=11)
    ax_bar.set_title(f'Dopant: {dopant}  |  Feature Importance (group avg)', fontsize=11)
    ax_bar.grid(axis='x', linestyle='--', alpha=0.4)
    for xi, yi in zip(mean_abs_group[order], range(len(feat_names))):
        ax_bar.text(xi + mean_abs_group.max() * 0.01, yi,
                    f'{xi:.4f}', va='center', fontsize=8)

    plt.suptitle(
        f'SHAP Analysis – Dopant System: {dopant} in Ni–Al base\n'
        f'Pooled across {n_alloys} compositions ({n_rows_grp} total training rows)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()

    safe_dop = dopant.replace('/', '_')
    fig_path = f'xgboost_results/shap_per_dopant/shap_dopant_{safe_dop}.png'
    plt.savefig(fig_path, dpi=130, bbox_inches='tight')
    plt.close()
    print(f"    → Saved: {fig_path}")

print("\n[Per-dopant SHAP figures saved — one plot per dopant group]")
print("[Continuing to composite cross-dopant figures ...]")


In [ ]:
print("=" * 70)
print("CELL 16b – CROSS-DOPANT SHAP COMPOSITE FIGURES")
print("=" * 70)

dopants_list = list(dopant_system_summary.keys())
n_dop        = len(dopants_list)


shap_mat = np.array([dopant_system_summary[d] for d in dopants_list])


shap_mat_norm = shap_mat / (shap_mat.sum(axis=1, keepdims=True) + 1e-12)

fig, ax = plt.subplots(figsize=(max(12, len(feat_names) * 1.8),
                                max(5, n_dop * 0.9)))
im = ax.imshow(shap_mat_norm, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Normalised mean |SHAP|')
ax.set_xticks(range(len(feat_names)))
ax.set_xticklabels(feat_names, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(n_dop))
ax.set_yticklabels(dopants_list, fontsize=11, fontweight='bold')
for i in range(n_dop):
    for j in range(len(feat_names)):
        ax.text(j, i, f'{shap_mat_norm[i, j]:.2f}',
                ha='center', va='center', fontsize=7.5,
                color='black' if shap_mat_norm[i, j] < 0.6 else 'white')
ax.set_title(
    'SHAP Heatmap – Per Dopant System (Ni–Al base)\n'
    'Normalised mean |SHAP|: brighter = more important for that dopant group',
    fontsize=12, fontweight='bold')
plt.tight_layout()
hm_path = 'xgboost_results/shap_dopant_heatmap.png'
plt.savefig(hm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {hm_path}")


top_feat_idx   = shap_mat.argmax(axis=1)
top_feat_vals  = shap_mat.max(axis=1)
top_feat_names = [feat_names[i] for i in top_feat_idx]

feat_color_map = {f: plt.cm.tab10(k / len(feat_names))
                  for k, f in enumerate(feat_names)}
bar_colors = [feat_color_map[f] for f in top_feat_names]

fig, ax = plt.subplots(figsize=(max(8, n_dop * 1.2), 6))
bars = ax.bar(dopants_list, top_feat_vals, color=bar_colors,
              edgecolor='k', linewidth=0.7)
ax.set_xlabel('Dopant System', fontsize=12)
ax.set_ylabel('Mean |SHAP| of dominant feature', fontsize=12)
ax.set_title('Most Important Feature per Dopant Group (SHAP)\n'
             'Ni–Al base doped with Pt / Pd / Ir / Rh',
             fontsize=13, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
for bar, fname, val in zip(bars, top_feat_names, top_feat_vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + top_feat_vals.max() * 0.02,
            fname, ha='center', va='bottom', fontsize=9, fontweight='bold')
handles = [plt.Rectangle((0,0), 1, 1, color=feat_color_map[f]) for f in feat_names]
ax.legend(handles, feat_names, title='Feature', fontsize=8, ncol=2)
plt.tight_layout()
bar_path = 'xgboost_results/shap_dopant_top_feature.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {bar_path}")


global_shap = shap_mat.mean(axis=0)
order_g     = np.argsort(global_shap)[::-1]
colors_g    = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(feat_names)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([feat_names[i] for i in order_g], global_shap[order_g],
       color=colors_g, edgecolor='k', linewidth=0.5)
ax.set_ylabel('Global Mean |SHAP|  (averaged over all dopant groups)', fontsize=11)
ax.set_xlabel('Feature', fontsize=11)
ax.set_title('Global Feature Importance – Ni–Al + PGM Dopants\n'
             '(pooled across Pt, Pd, Ir, Rh groups)', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=40)
ax.grid(axis='y', linestyle='--', alpha=0.4)
for i, (f_i, v) in enumerate(zip(order_g, global_shap[order_g])):
    ax.text(i, v + global_shap.max() * 0.01, f'{v:.4f}', ha='center', fontsize=8)
plt.tight_layout()
glob_path = 'xgboost_results/shap_dopant_global_bar.png'
plt.savefig(glob_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {glob_path}")


fig, ax = plt.subplots(figsize=(max(10, n_dop * 1.4), 7))
x_pos   = np.arange(n_dop)
bottom  = np.zeros(n_dop)
feat_colors_list = [plt.cm.tab10(k / len(feat_names)) for k in range(len(feat_names))]

for j, feat in enumerate(feat_names):
    vals = shap_mat[:, j]
    ax.bar(x_pos, vals, bottom=bottom,
           label=feat, color=feat_colors_list[j],
           edgecolor='k', linewidth=0.4)
    # value labels
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > shap_mat.max() * 0.03:   # only label if segment is large enough
            ax.text(xi, b + v / 2, f'{v:.3f}',
                    ha='center', va='center', fontsize=6.5, color='white',
                    fontweight='bold')
    bottom += vals

ax.set_xticks(x_pos)
ax.set_xticklabels(dopants_list, fontsize=13, fontweight='bold')
ax.set_ylabel('Mean |SHAP| (stacked)', fontsize=12)
ax.set_title('Stacked SHAP Feature Contributions – Per Dopant System\n'
             'Ni–Al base  |  Pt / Pd / Ir / Rh dopants',
             fontsize=13, fontweight='bold')
ax.legend(title='Feature', fontsize=9, loc='upper right',
          ncol=2, bbox_to_anchor=(1.16, 1))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
stack_path = 'xgboost_results/shap_dopant_stacked.png'
plt.savefig(stack_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {stack_path}")


x = np.arange(len(feat_names))
width = 0.8 / n_dop
dop_colors = plt.cm.Set1(np.linspace(0, 0.8, n_dop))

fig, ax = plt.subplots(figsize=(max(14, len(feat_names) * 2.2), 6))
for k, (dop, color) in enumerate(zip(dopants_list, dop_colors)):
    offset = (k - n_dop / 2 + 0.5) * width
    bars_g = ax.bar(x + offset, shap_mat[k], width,
                    label=dop, color=color, edgecolor='k', linewidth=0.4)
ax.set_xticks(x)
ax.set_xticklabels(feat_names, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Mean |SHAP|', fontsize=12)
ax.set_title('Feature Importance Comparison Across Dopant Systems\n'
             'Ni–Al base  |  Each colour = one PGM dopant group',
             fontsize=13, fontweight='bold')
ax.legend(title='Dopant', fontsize=10, ncol=n_dop)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
grp_path = 'xgboost_results/shap_dopant_grouped_bar.png'
plt.savefig(grp_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {grp_path}")


shap_csv = pd.DataFrame(
    shap_mat, columns=feat_names,
    index=pd.Index(dopants_list, name='dopant_system')
)
shap_csv['dominant_feature'] = top_feat_names
shap_csv.to_csv('xgboost_results/shap_dopant_system_summary.csv')
print("Saved -> xgboost_results/shap_dopant_system_summary.csv")

print("\n" + "=" * 70)
print("SHAP DOPANT-SYSTEM ANALYSIS COMPLETE")
print("=" * 70)
print("Generated files:")
print("  xgboost_results/shap_per_dopant/shap_dopant_Pt.png  (and Pd, Ir, Rh)")
print("  xgboost_results/shap_dopant_heatmap.png")
print("  xgboost_results/shap_dopant_top_feature.png")
print("  xgboost_results/shap_dopant_global_bar.png")
print("  xgboost_results/shap_dopant_stacked.png")
print("  xgboost_results/shap_dopant_grouped_bar.png")
print("  xgboost_results/shap_dopant_system_summary.csv")
